# [가설검증] 잠실 리센츠와 비슷한 가격의 강남·서초 아파트, 어느 쪽이 더 올랐을까?

84㎡와 59㎡ 후보를 나누어 단지별 장기 상승률과 동일 기간 리센츠 대비 초과상승률을 계산합니다.


In [14]:
# @title 상승률 비교표를 생성하세요
"""리센츠 가격대 강남·서초 아파트의 장기 상승률을 비교한다.

84㎡와 59㎡ 후보를 분리해 KB 일반가가 처음 함께 존재하는 월부터
2026년 7월까지 단지 CAGR과 동일 기간 리센츠 CAGR을 계산한다.
분양가와 분양권 가격은 사용하지 않고 복수 타입은 월별 중위값으로 통합한다.
"""

from __future__ import annotations

import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLBACKEND", "Agg")
matplotlib_cache = Path("/tmp/va_recenz_matplotlib")
matplotlib_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(matplotlib_cache))

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager
from IPython.display import FileLink, HTML, Image, Markdown, display


REFRESH_HISTORICAL_DATA = False  # @param {type:"boolean"}

API_PRICE = "https://api.kbland.kr/land-price"
REFERENCE_COMPLEX_ID = 15524
START_YEAR_MONTH = "200801"
END_YEAR_MONTH = "202607"
START_YEAR = 2008
END_YEAR = 2026
MIN_COMPARISON_MONTHS = 60
IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
HISTORY_PATH = OUTPUT_DIR / "kb_candidate_monthly_prices_2008_2026.csv"
SUMMARY_PATHS = {
    "84": OUTPUT_DIR / "recenz_vs_gangnam_84_growth.csv",
    "59": OUTPUT_DIR / "recenz_vs_gangnam_59_growth.csv",
}
DONG_SUMMARY_PATH = OUTPUT_DIR / "recenz_vs_gangnam_dong_summary.csv"
DONG_CHART_PATH = OUTPUT_DIR / "recenz_vs_gangnam_dong_growth.png"
YEAR_SUMMARY_PATH = OUTPUT_DIR / "recenz_vs_gangnam_year_summary.csv"
YEAR_CHART_PATH = OUTPUT_DIR / "recenz_vs_gangnam_year_growth.png"
HOUSEHOLD_SUMMARY_PATH = OUTPUT_DIR / "recenz_vs_gangnam_household_summary.csv"
HOUSEHOLD_CHART_PATH = OUTPUT_DIR / "recenz_vs_gangnam_household_growth.png"
INPUT_PATHS = {
    "84": {
        "candidates": OUTPUT_DIR / "candidates_84.csv",
        "types": OUTPUT_DIR / "types_84.csv",
    },
    "59": {
        "candidates": OUTPUT_DIR / "candidates_59.csv",
        "types": OUTPUT_DIR / "types_59.csv",
    },
}


def build_session():
    """KB 공개 API 요청용 세션을 만든다."""
    import requests

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "ko-KR,ko;q=0.9",
        "Origin": "https://kbland.kr",
        "Referer": "https://kbland.kr/",
    })
    return session


def request_year_prices(
    complex_id: int, area_id: int, year: int
) -> list[dict[str, Any]]:
    """한 면적 타입의 특정 연도 월별 KB 일반가를 반환한다."""
    session = build_session()
    last_error = None
    for attempt in range(7):
        try:
            response = session.get(
                f"{API_PRICE}/price/WholQuotList",
                params={
                    "단지기본일련번호": complex_id,
                    "면적일련번호": area_id,
                    "기준년": str(year),
                },
                timeout=30,
            )
            response.raise_for_status()
            data = response.json().get("dataBody", {}).get("data", {})
            groups = data.get("시세", []) if isinstance(data, dict) else []
            rows = []
            for group in groups:
                for item in group.get("items", []):
                    year_month = str(item.get("기준년월") or "")
                    raw_price = item.get("매매일반거래가")
                    if (
                        len(year_month) == 6
                        and START_YEAR_MONTH <= year_month <= END_YEAR_MONTH
                        and raw_price
                    ):
                        rows.append({
                            "단지기본일련번호": complex_id,
                            "면적일련번호": area_id,
                            "기준년월": year_month,
                            "KB일반가_만원": int(raw_price),
                        })
            return rows
        except Exception as error:
            last_error = error
            time.sleep(0.8 * (attempt + 1))
    raise RuntimeError(f"KB 가격 조회 실패: {complex_id}/{area_id}/{year}") from last_error


def load_inputs() -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """면적별 후보와 해당 후보의 전체 면적 타입을 불러온다."""
    candidates_by_size = {}
    selected_types = []
    for size, paths in INPUT_PATHS.items():
        if not paths["candidates"].exists() or not paths["types"].exists():
            raise FileNotFoundError(f"{size}㎡ 입력 CSV가 없습니다.")
        candidates = pd.read_csv(
            paths["candidates"], dtype={"입주년월": str}
        )
        candidates["단지기본일련번호"] = (
            candidates["KB링크"].str.extract(r"(\d+)$")[0].astype(int)
        )
        types = pd.read_csv(paths["types"])
        types = types[
            types["단지기본일련번호"].isin(candidates["단지기본일련번호"])
        ].copy()
        types["면적구분"] = size
        candidates_by_size[size] = candidates
        selected_types.append(types)
    return candidates_by_size, pd.concat(selected_types, ignore_index=True)


def collect_history(selected_types: pd.DataFrame) -> pd.DataFrame:
    """후보 단지 면적 타입의 2008~2026년 월별 KB 일반가를 수집한다."""
    type_keys = selected_types[
        ["면적구분", "단지기본일련번호", "면적일련번호"]
    ].drop_duplicates()
    tasks = [
        (row["면적구분"], int(row["단지기본일련번호"]),
         int(row["면적일련번호"]), year)
        for _, row in type_keys.iterrows()
        for year in range(START_YEAR, END_YEAR + 1)
    ]
    rows = []
    failures = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {
            executor.submit(request_year_prices, complex_id, area_id, year):
            (size, complex_id, area_id, year)
            for size, complex_id, area_id, year in tasks
        }
        for completed, future in enumerate(as_completed(futures), 1):
            size, complex_id, area_id, year = futures[future]
            try:
                result = future.result()
                rows.extend({**item, "면적구분": size} for item in result)
            except RuntimeError:
                failures.append((size, complex_id, area_id, year))
            if completed % 250 == 0 or completed == len(tasks):
                print(f"과거 가격 조회: {completed:,}/{len(tasks):,}", flush=True)
    for size, complex_id, area_id, year in failures:
        result = request_year_prices(complex_id, area_id, year)
        rows.extend({**item, "면적구분": size} for item in result)
        time.sleep(0.2)
    history = pd.DataFrame(rows).drop_duplicates(
        ["면적구분", "단지기본일련번호", "면적일련번호", "기준년월"]
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    history.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    return history


def aggregate_monthly_prices(history: pd.DataFrame) -> pd.DataFrame:
    """복수 면적 타입을 단지별 월간 일반가 중위값으로 합친다."""
    return (
        history.groupby(
            ["면적구분", "단지기본일련번호", "기준년월"], as_index=False
        )["KB일반가_만원"]
        .median()
        .sort_values(["면적구분", "단지기본일련번호", "기준년월"])
    )


def calculate_size_results(
    size: str, candidates: pd.DataFrame, monthly: pd.DataFrame
) -> pd.DataFrame:
    """단지별 CAGR과 동일 기간 리센츠 대비 초과상승률을 계산한다."""
    size_monthly = monthly[monthly["면적구분"].astype(str).eq(size)]
    reference = size_monthly[
        size_monthly["단지기본일련번호"].eq(REFERENCE_COMPLEX_ID)
    ].set_index("기준년월")["KB일반가_만원"]
    if END_YEAR_MONTH not in reference.index:
        raise ValueError(f"리센츠 {size}㎡의 2026년 7월 가격이 없습니다.")
    rows = []
    for _, candidate in candidates.iterrows():
        complex_id = int(candidate["단지기본일련번호"])
        prices = size_monthly[
            size_monthly["단지기본일련번호"].eq(complex_id)
        ].set_index("기준년월")["KB일반가_만원"]
        common_months = sorted(
            set(prices.index).intersection(reference.index)
            & {month for month in prices.index if START_YEAR_MONTH <= month <= END_YEAR_MONTH}
        )
        if not common_months or END_YEAR_MONTH not in common_months:
            raise ValueError(f"{candidate['아파트']}의 공통 비교기간이 없습니다.")
        start_month = common_months[0]
        start_period = pd.Period(start_month, freq="M")
        end_period = pd.Period(END_YEAR_MONTH, freq="M")
        months = (end_period.year - start_period.year) * 12 + (
            end_period.month - start_period.month
        )
        if months <= 0:
            raise ValueError(f"{candidate['아파트']}의 비교기간이 너무 짧습니다.")
        if months < MIN_COMPARISON_MONTHS and complex_id != REFERENCE_COMPLEX_ID:
            continue
        years = months / 12
        start_price = float(prices.loc[start_month])
        end_price = float(prices.loc[END_YEAR_MONTH])
        reference_start = float(reference.loc[start_month])
        reference_end = float(reference.loc[END_YEAR_MONTH])
        cagr = (end_price / start_price) ** (1 / years) - 1
        reference_cagr = (reference_end / reference_start) ** (1 / years) - 1
        rows.append({
            "단지기본일련번호": complex_id,
            "자치구": candidate["자치구"],
            "동": candidate["동"],
            "아파트": candidate["아파트"],
            "KB링크": candidate["KB링크"],
            "준공연도": candidate["준공연도"],
            "비교시작월": start_month,
            "비교종료월": END_YEAR_MONTH,
            "비교개월": months,
            "시작가격_억원": start_price / 10_000,
            "현재가격_억원": end_price / 10_000,
            "단지_CAGR": cagr,
            "리센츠_CAGR": reference_cagr,
            "초과상승률": cagr - reference_cagr,
            "기준행": bool(candidate["기준행"]),
        })
    result = pd.DataFrame(rows)
    reference_rows = result[result["기준행"]]
    comparison_rows = result[~result["기준행"]].sort_values(
        ["초과상승률", "아파트"], ascending=[False, True]
    )
    return pd.concat([reference_rows, comparison_rows], ignore_index=True)


def combine_dong_results(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """두 면적 결과의 중복 단지를 합쳐 동별 초과상승률을 요약한다."""
    observations = []
    for size, result in results.items():
        comparison = result[~result["기준행"]].copy()
        comparison["면적구분"] = size
        observations.append(comparison)
    combined = pd.concat(observations, ignore_index=True)
    complex_results = (
        combined.groupby(
            ["단지기본일련번호", "자치구", "동", "아파트"],
            as_index=False,
        )
        .agg(
            포함면적=("면적구분", lambda values: "·".join(sorted(values))),
            초과상승률=("초과상승률", "median"),
        )
    )
    summary = (
        complex_results.groupby(["자치구", "동"], as_index=False)
        .agg(
            비교단지수=("아파트", "size"),
            리센츠초과단지=("초과상승률", lambda values: int(values.gt(0).sum())),
            초과상승률중앙값=("초과상승률", "median"),
        )
    )
    summary["승률"] = summary["리센츠초과단지"] / summary["비교단지수"]

    def classify(row: pd.Series) -> str:
        if row["비교단지수"] < 2:
            return "참고"
        excess = row["초과상승률중앙값"]
        if excess >= 0.005:
            return f"{row['동']} 우세"
        if excess > 0:
            return f"{row['동']} 소폭 우세"
        if excess <= -0.005:
            return "리센츠 우세"
        if excess < 0:
            return "리센츠 소폭 우세"
        return "비슷"

    summary["판정"] = summary.apply(classify, axis=1)
    return summary.sort_values(
        ["초과상승률중앙값", "동"], ascending=[False, True]
    ).reset_index(drop=True)


def combine_year_results(results: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """두 면적 결과의 중복 단지를 합쳐 준공연도 구간별로 요약한다."""
    observations = []
    for size, result in results.items():
        comparison = result[~result["기준행"]].copy()
        comparison["면적구분"] = size
        observations.append(comparison)
    combined = pd.concat(observations, ignore_index=True)
    complex_results = (
        combined.groupby(
            ["단지기본일련번호", "아파트", "준공연도"],
            as_index=False,
        )
        .agg(
            포함면적=("면적구분", lambda values: "·".join(sorted(values))),
            초과상승률=("초과상승률", "median"),
        )
    )
    complex_results["연식그룹"] = pd.cut(
        complex_results["준공연도"],
        bins=[0, 1999, 2009, 2019],
        labels=["1999년 이전", "2000~2009년", "2010~2019년"],
        include_lowest=True,
    )
    summary = (
        complex_results.groupby("연식그룹", observed=True, as_index=False)
        .agg(
            비교단지수=("아파트", "size"),
            리센츠초과단지=("초과상승률", lambda values: int(values.gt(0).sum())),
            초과상승률중앙값=("초과상승률", "median"),
        )
    )
    summary["승률"] = summary["리센츠초과단지"] / summary["비교단지수"]
    return summary


def combine_household_results(
    results: dict[str, pd.DataFrame],
    candidates_by_size: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    """두 면적 결과를 합쳐 세대수 구간별 초과상승률을 요약한다."""
    observations = []
    for size, result in results.items():
        comparison = result[~result["기준행"]].copy()
        households = candidates_by_size[size][
            ["단지기본일련번호", "세대수"]
        ].drop_duplicates("단지기본일련번호")
        comparison = comparison.merge(
            households, on="단지기본일련번호", how="left", validate="many_to_one"
        )
        comparison["면적구분"] = size
        observations.append(comparison)
    combined = pd.concat(observations, ignore_index=True)
    complex_results = (
        combined.groupby(
            ["단지기본일련번호", "아파트", "세대수"], as_index=False
        )
        .agg(
            포함면적=("면적구분", lambda values: "·".join(sorted(values))),
            초과상승률=("초과상승률", "median"),
        )
    )
    complex_results["세대수그룹"] = pd.cut(
        complex_results["세대수"],
        bins=[0, 499, 799, float("inf")],
        labels=["500세대 미만", "500~799세대", "800세대 이상"],
        include_lowest=True,
    )
    summary = (
        complex_results.groupby("세대수그룹", observed=True, as_index=False)
        .agg(
            비교단지수=("아파트", "size"),
            리센츠초과단지=("초과상승률", lambda values: int(values.gt(0).sum())),
            초과상승률중앙값=("초과상승률", "median"),
        )
    )
    summary["승률"] = summary["리센츠초과단지"] / summary["비교단지수"]
    return summary


def format_price(price_eok: float) -> str:
    """억원 단위 값을 소수점 둘째 자리까지 표시한다."""
    number = f"{price_eok:.2f}".rstrip("0").rstrip(".")
    return f"{number}억원"


def format_period(start_month: str, end_month: str) -> str:
    """YYYYMM 두 값을 읽기 쉬운 비교기간으로 표시한다."""
    return f"{start_month[:4]}.{int(start_month[4:])}~{end_month[:4]}.{int(end_month[4:])}"


def build_table_html(size: str, result: pd.DataFrame) -> str:
    """면적별 상승률 비교 HTML 표를 만든다."""
    rows = []
    for _, row in result.iterrows():
        row_class = " class='reference-row'" if row["기준행"] else ""
        rows.append(
            f"<tr{row_class}>"
            f"<td class='name'><a href='{row['KB링크']}' target='_blank' "
            f"rel='noopener noreferrer'><span style='white-space:nowrap;word-break:keep-all'>{row['아파트']}</span></a></td>"
            f"<td class='center'><span style='white-space:nowrap'>{int(row['준공연도'])}</span></td>"
            f"<td class='center'><span style='white-space:nowrap'>{format_period(row['비교시작월'], row['비교종료월'])}</span></td>"
            f"<td class='number'><span style='white-space:nowrap'>{format_price(row['시작가격_억원'])}</span></td>"
            f"<td class='number'><span style='white-space:nowrap'>{format_price(row['현재가격_억원'])}</span></td>"
            f"<td class='number'><span style='white-space:nowrap'>{row['단지_CAGR']:.2%}</span></td>"
            f"<td class='number'><span style='white-space:nowrap'>{row['리센츠_CAGR']:.2%}</span></td>"
            f"<td class='number'><span style='white-space:nowrap'>{row['초과상승률']:+.2%}p</span></td>"
            "</tr>"
        )
    comparison = result[~result["기준행"]]
    win_count = int(comparison["초과상승률"].gt(0).sum())
    median_excess = float(comparison["초과상승률"].median())
    return f"""
<section class='growth-section'>
  <p class='brand'>대도시 연구실</p>
  <h2>리센츠 가격대 아파트 상승률 | 전용 {size}㎡</h2>
  <p class='subtitle'>KB 일반가 · 단지별 최초 공통월~2026년 7월 · 동일 기간 리센츠 비교</p>
  <p class='summary'>리센츠 초과 단지 <strong>{win_count}/{len(comparison)}</strong> · 초과상승률 중앙값 <strong>{median_excess:+.2%}p</strong></p>
  <div class='table-wrap'><table>
    <thead><tr><th>단지</th><th>준공</th><th>비교기간</th><th>시작가격</th><th>현재가격</th><th>단지 연평<br>균상승률</th><th>리센츠 연<br>평균상승률</th><th>초과상승률</th></tr></thead>
    <tbody>{''.join(rows)}</tbody>
  </table></div>
  <p class='footnote'>※ 단기 가격 변동의 영향을 줄이기 위해 최소 5년 이상의 비교기간이 확보된 단지만 분석<br>※ 복수 타입은 월별 KB 일반가 중위값으로 통합<br>※ 2026년 7월 리센츠 가격대 단지를 현재 시점에서 선별한 뒤 과거 상승률을 비교한 결과<br>※ 초과상승률(%p) = 해당 단지 연평균상승률 - 리센츠 연평균상승률 (동일 기간 기준)</p>
</section>
"""


def build_style_html() -> str:
    """두 결과표에 공통으로 사용할 시각 스타일을 반환한다."""
    return """
<style>
.growth-section { max-width:700px; margin:14px 0 34px; font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; }
.growth-section .brand { margin:0 0 5px; color:#64748b; font-size:13px; }
.growth-section h2 { margin:0 0 4px; color:#0b0b0b; font-size:20px; line-height:1.3; }
.growth-section .subtitle { margin:0 0 14px; color:#64748b; font-size:13px; }
.growth-section .summary { margin:0 0 12px; color:#1e293b; font-size:13px; }
.growth-section .table-wrap { overflow:hidden; border:1px solid #f0f2f5; border-radius:12px; }
.growth-section table { width:700px; table-layout:fixed; border-collapse:separate; border-spacing:0; color:#1e293b; font-size:13px; font-variant-numeric:tabular-nums; }
.growth-section th { padding:10px 1px; background:#2b4a75; color:white; text-align:center; font-size:13px; font-weight:700; line-height:1.3; white-space:normal; word-break:keep-all; }
.growth-section td { padding:10px 1px; border-bottom:1px solid #f0f2f5; white-space:nowrap !important; word-break:keep-all !important; line-height:1.4; }
.growth-section tbody tr:nth-child(even) td { background:#fafbfc; }
.growth-section tbody tr:last-child td { border-bottom:0; }
.growth-section .name { overflow:hidden; text-align:left; text-overflow:ellipsis; font-weight:600; }
.growth-section .name a { color:inherit; text-decoration:none; }
.growth-section .name a:hover { color:#2f7dd3; text-decoration:underline; }
.growth-section .center { text-align:center; }
.growth-section .number { text-align:right; }
.growth-section .reference-row td { background:#eaf2fc !important; border-top:2px solid #2f7dd3; border-bottom:2px solid #c5d9f4; font-weight:700; }
.growth-section th:nth-child(1) { width:21%; }
.growth-section th:nth-child(2) { width:9%; }
.growth-section th:nth-child(3) { width:16%; }
.growth-section th:nth-child(4) { width:11%; }
.growth-section th:nth-child(5) { width:12%; }
.growth-section th:nth-child(6) { width:10.5%; }
.growth-section th:nth-child(7) { width:11%; }
.growth-section th:nth-child(8) { width:9.5%; }
.growth-section .footnote { margin:9px 0 0; color:#64748b; font-size:13px; line-height:1.5; }
</style>
"""


def select_font_family() -> str:
    """사용 가능한 한글 글꼴을 우선순위에 따라 선택한다."""
    available = {font.name for font in font_manager.fontManager.ttflist}
    for candidate in (
        "Pretendard", "Apple SD Gothic Neo",
        "Noto Sans CJK KR", "Malgun Gothic",
    ):
        if candidate in available:
            return candidate
    return "DejaVu Sans"


def create_dong_summary_chart(summary: pd.DataFrame) -> Path:
    """동별 초과상승률 중앙값을 가로 발산형 막대그래프로 저장한다."""
    plot_data = summary.sort_values("초과상승률중앙값").reset_index(drop=True)
    values = plot_data["초과상승률중앙값"] * 100
    colors = ["#2F7DD3" if value > 0 else "#F06432" for value in values]
    font_family = select_font_family()
    plt.rcParams["font.family"] = font_family
    plt.rcParams["axes.unicode_minus"] = False

    fig = plt.figure(figsize=(8, 9), facecolor="#FFFFFF")
    ax = fig.add_axes([0.16, 0.32, 0.79, 0.49])
    bars = ax.barh(plot_data["동"], values, color=colors, height=0.62)
    limit = max(abs(values.min()), abs(values.max())) * 1.72
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color="#777777", linewidth=1.2)
    ax.grid(axis="x", color="#DEDCD6", linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#DEDCD6")
    ax.tick_params(axis="x", colors="#777777", labelsize=14, length=0)
    ax.tick_params(axis="y", colors="#777777", labelsize=14, length=0, pad=22)
    ax.set_xlabel(
        "리센츠 대비 연평균 초과상승률(%p)",
        fontsize=15, color="#777777", labelpad=12,
    )
    ax.text(0.01, 1.025, "← 리센츠 우세", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=13, color="#F06432")
    ax.text(0.99, 1.025, "동 우세 →", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=13, color="#2F7DD3")

    offset = limit * 0.025
    for bar, (_, row), value, color in zip(
        bars, plot_data.iterrows(), values, colors
    ):
        count_text = f"{int(row['리센츠초과단지'])}/{int(row['비교단지수'])}"
        label = f"{value:+.2f}%p ({count_text})"
        ax.text(
            value + (offset if value >= 0 else -offset),
            bar.get_y() + bar.get_height() / 2,
            label, ha="left" if value >= 0 else "right", va="center",
            fontsize=13, fontweight="bold", color=color,
        )

    total_complexes = int(summary["비교단지수"].sum())
    fig.text(0.02, 0.965, "대도시 연구실", fontsize=13, color="#64748B")
    fig.text(
        0.02, 0.915, "리센츠 대비 동별 상승률 비교 | 전용 59·84㎡ 종합",
        fontsize=21, fontweight="bold", color="#0B0B0B",
    )
    fig.text(
        0.02, 0.875,
        "최소 5년 비교 · 동일 단지 중복 통합 · 초과상승률 중앙값 기준",
        fontsize=16, color="#64748B",
    )
    footnote = (
        f"※ 최소 5년 비교기간을 충족한 59·84㎡ 중복 통합 {total_complexes}개 단지 집계\n"
        "※ 동일 단지가 두 면적에 포함되면 면적별 초과상승률 중앙값을 대표값으로 사용\n"
        "※ 초과상승률(%p) = 해당 단지 연평균상승률 - 리센츠 연평균상승률 (동일 기간 동일 면적 기준)\n"
        "※ 막대 라벨: 초과상승률 중앙값 (리센츠 초과 단지 수/비교 단지 수)"
    )
    fig.text(
        0.02, 0.21, footnote, fontsize=13, color="#64748B",
        linespacing=1.22, va="top",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(DONG_CHART_PATH, dpi=200, bbox_inches="tight", facecolor="#FFFFFF")
    plt.close(fig)
    return DONG_CHART_PATH


def create_year_summary_chart(summary: pd.DataFrame) -> Path:
    """준공연도 구간별 초과상승률 중앙값을 발산형 막대그래프로 저장한다."""
    plot_data = summary.copy()
    values = plot_data["초과상승률중앙값"] * 100
    colors = ["#2F7DD3" if value > 0 else "#F06432" for value in values]
    font_family = select_font_family()
    plt.rcParams["font.family"] = font_family
    plt.rcParams["axes.unicode_minus"] = False

    fig = plt.figure(figsize=(8, 6.5), facecolor="#FFFFFF")
    ax = fig.add_axes([0.20, 0.38, 0.75, 0.30])
    bars = ax.barh(plot_data["연식그룹"].astype(str), values, color=colors, height=0.54)
    ax.invert_yaxis()
    limit = max(abs(values.min()), abs(values.max())) * 1.72
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color="#777777", linewidth=1.2)
    ax.grid(axis="x", color="#DEDCD6", linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#DEDCD6")
    ax.tick_params(axis="x", colors="#777777", labelsize=14, length=0)
    ax.tick_params(axis="y", colors="#777777", labelsize=14, length=0, pad=22)
    ax.set_xlabel(
        "리센츠 대비 연평균 초과상승률(%p)",
        fontsize=15, color="#777777", labelpad=12,
    )
    ax.text(0.01, 1.08, "← 리센츠 우세", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=13, color="#F06432")
    ax.text(0.99, 1.08, "연식 그룹 우세 →", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=13, color="#2F7DD3")

    offset = limit * 0.025
    for bar, (_, row), value, color in zip(
        bars, plot_data.iterrows(), values, colors
    ):
        count_text = f"{int(row['리센츠초과단지'])}/{int(row['비교단지수'])}"
        label = f"{value:+.2f}%p ({count_text})"
        ax.text(
            value + (offset if value >= 0 else -offset),
            bar.get_y() + bar.get_height() / 2,
            label, ha="left" if value >= 0 else "right", va="center",
            fontsize=13, fontweight="bold", color=color,
        )

    fig.text(0.02, 0.95, "대도시 연구실", fontsize=13, color="#64748B")
    fig.text(
        0.02, 0.885, "리센츠 대비 준공연도별 상승률 비교 | 전용 59·84㎡ 종합",
        fontsize=20, fontweight="bold", color="#0B0B0B",
    )
    fig.text(
        0.02, 0.835,
        "최소 5년 비교 · 동일 단지 중복 통합 · 초과상승률 중앙값 기준",
        fontsize=14.5, color="#64748B",
    )
    total_complexes = int(summary["비교단지수"].sum())
    footnote = (
        f"※ 분석 대상 {total_complexes}개 단지를 1999년 이전·2000년대·2010년대로 구분\n"
        "※ 동일 단지가 두 면적에 포함되면 면적별 초과상승률 중앙값을 대표값으로 사용\n"
        "※ 초과상승률(%p) = 해당 단지 연평균상승률 - 리센츠 연평균상승률 (동일 기간 동일 면적 기준)\n"
        "※ 막대 라벨: 초과상승률 중앙값 (리센츠 초과 단지 수/비교 단지 수)"
    )
    fig.text(
        0.02, 0.26, footnote, fontsize=12.5, color="#64748B",
        linespacing=1.22, va="top",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(YEAR_CHART_PATH, dpi=200, bbox_inches="tight", facecolor="#FFFFFF")
    plt.close(fig)
    return YEAR_CHART_PATH


def create_household_summary_chart(summary: pd.DataFrame) -> Path:
    """세대수 구간별 초과상승률 중앙값을 발산형 막대그래프로 저장한다."""
    plot_data = summary.copy()
    values = plot_data["초과상승률중앙값"] * 100
    colors = ["#2F7DD3" if value > 0 else "#F06432" for value in values]
    font_family = select_font_family()
    plt.rcParams["font.family"] = font_family
    plt.rcParams["axes.unicode_minus"] = False

    fig = plt.figure(figsize=(8, 6.5), facecolor="#FFFFFF")
    ax = fig.add_axes([0.20, 0.38, 0.75, 0.30])
    bars = ax.barh(
        plot_data["세대수그룹"].astype(str), values, color=colors, height=0.54
    )
    ax.invert_yaxis()
    limit = max(abs(values.min()), abs(values.max())) * 2.0
    ax.set_xlim(-limit, limit)
    ax.axvline(0, color="#777777", linewidth=1.2)
    ax.grid(axis="x", color="#DEDCD6", linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#DEDCD6")
    ax.tick_params(axis="x", colors="#777777", labelsize=14, length=0)
    ax.tick_params(axis="y", colors="#777777", labelsize=14, length=0, pad=22)
    ax.set_xlabel(
        "리센츠 대비 연평균 초과상승률(%p)",
        fontsize=15, color="#777777", labelpad=12,
    )
    ax.text(0.01, 1.08, "← 리센츠 우세", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=13, color="#F06432")
    ax.text(0.99, 1.08, "세대수 그룹 우세 →", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=13, color="#2F7DD3")

    offset = limit * 0.025
    for bar, (_, row), value, color in zip(
        bars, plot_data.iterrows(), values, colors
    ):
        count_text = f"{int(row['리센츠초과단지'])}/{int(row['비교단지수'])}"
        label = f"{value:+.2f}%p ({count_text})"
        ax.text(
            value + (offset if value >= 0 else -offset),
            bar.get_y() + bar.get_height() / 2,
            label, ha="left" if value >= 0 else "right", va="center",
            fontsize=13, fontweight="bold", color=color,
        )

    fig.text(0.02, 0.95, "대도시 연구실", fontsize=13, color="#64748B")
    fig.text(
        0.02, 0.885, "리센츠 대비 세대수별 상승률 비교 | 전용 59·84㎡ 종합",
        fontsize=20, fontweight="bold", color="#0B0B0B",
    )
    fig.text(
        0.02, 0.835,
        "최소 5년 비교 · 동일 단지 중복 통합 · 초과상승률 중앙값 기준",
        fontsize=14.5, color="#64748B",
    )
    total_complexes = int(summary["비교단지수"].sum())
    footnote = (
        f"※ 분석 대상 {total_complexes}개 단지를 500세대 미만·500~799세대·800세대 이상으로 구분\n"
        "※ 동일 단지가 두 면적에 포함되면 면적별 초과상승률 중앙값을 대표값으로 사용\n"
        "※ 초과상승률(%p) = 해당 단지 연평균상승률 - 리센츠 연평균상승률 (동일 기간 동일 면적 기준)\n"
        "※ 막대 라벨: 초과상승률 중앙값 (리센츠 초과 단지 수/비교 단지 수)"
    )
    fig.text(
        0.02, 0.26, footnote, fontsize=12.5, color="#64748B",
        linespacing=1.22, va="top",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        HOUSEHOLD_CHART_PATH, dpi=200, bbox_inches="tight", facecolor="#FFFFFF"
    )
    plt.close(fig)
    return HOUSEHOLD_CHART_PATH


def main() -> None:
    """과거 가격을 수집하거나 캐시를 읽어 두 면적 결과를 출력한다."""
    candidates_by_size, selected_types = load_inputs()
    if REFRESH_HISTORICAL_DATA is True:
        print("실행 모드: 과거 가격 새로 수집 ON", flush=True)
        history = collect_history(selected_types)
    elif HISTORY_PATH.exists():
        print("실행 모드: 과거 가격 새로 수집 OFF — 저장 CSV 사용", flush=True)
        history = pd.read_csv(HISTORY_PATH, dtype={"기준년월": str, "면적구분": str})
    else:
        display(HTML("""
<div style='max-width:700px;padding:16px 18px;border:1px solid #f3c7c7;border-radius:12px;background:#fff7f7;color:#991b1b'>
  <strong>저장된 과거 가격 결과가 없습니다.</strong><br>
  <code>과거 가격 새로 수집</code> 옵션을 ON으로 바꾼 뒤 실행해 주세요.
</div>"""))
        return
    monthly = aggregate_monthly_prices(history)
    results = {}
    for size in ("84", "59"):
        result = calculate_size_results(size, candidates_by_size[size], monthly)
        result.to_csv(SUMMARY_PATHS[size], index=False, encoding="utf-8-sig")
        results[size] = result
    dong_summary = combine_dong_results(results)
    dong_summary.to_csv(DONG_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    chart_path = create_dong_summary_chart(dong_summary)
    year_summary = combine_year_results(results)
    year_summary.to_csv(YEAR_SUMMARY_PATH, index=False, encoding="utf-8-sig")
    year_chart_path = create_year_summary_chart(year_summary)
    household_summary = combine_household_results(results, candidates_by_size)
    household_summary.to_csv(
        HOUSEHOLD_SUMMARY_PATH, index=False, encoding="utf-8-sig"
    )
    household_chart_path = create_household_summary_chart(household_summary)
    excluded_counts = {
        size: (
            int((~candidates_by_size[size]["기준행"]).sum())
            - int((~results[size]["기준행"]).sum())
        )
        for size in ("84", "59")
    }
    display(HTML(build_style_html() + build_table_html("84", results["84"])))
    display(Markdown(
        f"> ※ 비교기간 5년 미만으로 제외된 84㎡ 단지: **{excluded_counts['84']}개**"
    ))
    display(HTML(build_table_html("59", results["59"])))
    display(Markdown(
        f"> ※ 비교기간 5년 미만으로 제외된 59㎡ 단지: **{excluded_counts['59']}개**"
    ))
    total_complexes = int(household_summary["비교단지수"].sum())
    winning_complexes = int(household_summary["리센츠초과단지"].sum())
    winning_rate = winning_complexes / total_complexes
    combined_excess = pd.concat(
        [
            result.loc[~result["기준행"], ["단지기본일련번호", "초과상승률"]]
            for result in results.values()
        ],
        ignore_index=True,
    )
    median_excess = (
        combined_excess.groupby("단지기본일련번호")["초과상승률"]
        .median()
        .median()
    )
    display(Markdown(
        f"**59·84㎡ 중복 단지 통합 기준: 총 {total_complexes}개 단지 중 "
        f"리센츠 초과 {winning_complexes}개 ({winning_rate:.1%}) · "
        f"초과상승률 중앙값 {median_excess:+.2%}p**"
    ))
    display(HTML(
        f"<img src='{chart_path.as_posix()}' "
        "style='display:block;margin:0;width:auto;max-width:100%;' />"
    ))
    display(HTML(
        f"<img src='{year_chart_path.as_posix()}' "
        "style='display:block;margin:0;width:auto;max-width:100%;' />"
    ))
    display(HTML(
        f"<img src='{household_chart_path.as_posix()}' "
        "style='display:block;margin:0;width:auto;max-width:100%;' />"
    ))
    display(FileLink(str(SUMMARY_PATHS["84"]), result_html_prefix="84㎡ CSV: "))
    display(FileLink(str(SUMMARY_PATHS["59"]), result_html_prefix="59㎡ CSV: "))
    display(FileLink(str(DONG_SUMMARY_PATH), result_html_prefix="동별 종합 CSV: "))
    display(FileLink(str(chart_path), result_html_prefix="동별 종합 그래프: "))
    display(FileLink(str(YEAR_SUMMARY_PATH), result_html_prefix="준공연도별 종합 CSV: "))
    display(FileLink(str(year_chart_path), result_html_prefix="준공연도별 종합 그래프: "))
    display(FileLink(str(HOUSEHOLD_SUMMARY_PATH), result_html_prefix="세대수별 종합 CSV: "))
    display(FileLink(str(household_chart_path), result_html_prefix="세대수별 종합 그래프: "))


main()


실행 모드: 과거 가격 새로 수집 OFF — 저장 CSV 사용


단지,준공,비교기간,시작가격,현재가격,단지 연평균상승률,리센츠 연평균상승률,초과상승률
리센츠,2008,2008.1~2026.7,9.3억원,33.5억원,7.17%,7.17%,+0.00%p
신반포(한신16차),1983,2008.1~2026.7,7억원,33억원,8.74%,7.17%,+1.57%p
한강,1989,2008.1~2026.7,7.65억원,33.5억원,8.31%,7.17%,+1.14%p
잠원한신,1992,2008.1~2026.7,8.7억원,34억원,7.65%,7.17%,+0.47%p
삼성동힐스테이트1단지,2009,2008.1~2026.7,9.03억원,35억원,7.60%,7.17%,+0.43%p
삼성동힐스테이트2단지,2009,2008.1~2026.7,9.03억원,34억원,7.43%,7.17%,+0.26%p
진흥,1984,2008.1~2026.7,9.25억원,34억원,7.29%,7.17%,+0.12%p
우성7차(개포),1987,2008.1~2026.7,9.05억원,32.75억원,7.20%,7.17%,+0.03%p
삼성동중앙하이츠빌리지,2004,2008.1~2026.7,9.09억원,32억원,7.04%,7.17%,-0.13%p
테헤란아이파크,2014,2018.3~2026.7,16.65억원,32.5억원,8.36%,8.60%,-0.24%p


> ※ 비교기간 5년 미만으로 제외된 84㎡ 단지: **4개**

단지,준공,비교기간,시작가격,현재가격,단지 연평균상승률,리센츠 연평균상승률,초과상승률
리센츠,2008,2008.8~2026.7,6.65억원,29.75억원,8.72%,8.72%,+0.00%p
동아,2002,2008.8~2026.7,6.08억원,30.75억원,9.47%,8.72%,+0.75%p
대치아이파크,2007,2008.8~2026.7,7.38억원,30.5억원,8.25%,8.72%,-0.48%p
디에이치아너힐즈,2019,2020.10~2026.7,20억원,30.75억원,7.77%,8.61%,-0.84%p
반포써밋,2018,2018.9~2026.7,16.25억원,30.75억원,8.48%,9.85%,-1.37%p
래미안서초에스티지,2016,2016.11~2026.7,10.15억원,28.5억원,11.27%,12.78%,-1.51%p
래미안블레스티지,2019,2019.1~2026.7,16억원,28.5억원,8.00%,10.57%,-2.57%p
반포래미안아이파크,2018,2018.7~2026.7,17.5억원,31억원,7.41%,10.48%,-3.07%p
삼성동센트럴아이파크,2018,2018.3~2026.7,17억원,28.6억원,6.44%,9.56%,-3.12%p


> ※ 비교기간 5년 미만으로 제외된 59㎡ 단지: **4개**

...

/Users/1111429/src/daedosee-lab/va_recenz_vs_gamnam/output/recenz_vs_gangnam_dong_growth.png

/Users/1111429/src/daedosee-lab/va_recenz_vs_gamnam/output/recenz_vs_gangnam_year_summary.csv

/Users/1111429/src/daedosee-lab/va_recenz_vs_gamnam/output/recenz_vs_gangnam_year_growth.png

/Users/1111429/src/daedosee-lab/va_recenz_vs_gamnam/output/recenz_vs_gangnam_household_summary.csv

/Users/1111429/src/daedosee-lab/va_recenz_vs_gamnam/output/recenz_vs_gangnam_household_growth.png